https://www.kaggle.com/datasets/behrad3d/nasa-cmaps DATASET LINK THAT ARE USED IN CODE

MODEL 2 TASK 2

In [3]:

import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("train_FD001.txt", sep=" ", header=None)
print(df.head())
df.drop([26, 27], axis=1, inplace=True)  # Drop empty columns if present
df.columns = ['engine_no', 'cycle'] + [f'op_setting_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]

# Time-based feature: Time to failure
df['time_to_failure'] = df.groupby('engine_no')['cycle'].transform('max') - df['cycle']

# Rolling mean and std for selected sensors
rolling_cols = ['sensor_2', 'sensor_3', 'sensor_5']

for col in rolling_cols:
    df[f'{col}_rolling_mean'] = df.groupby('engine_no')[col].transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    df[f'{col}_rolling_std'] = df.groupby('engine_no')[col].transform(lambda x: x.rolling(window=5, min_periods=1).std())

# Lag features
for col in rolling_cols:
    df[f'{col}_lag1'] = df.groupby('engine_no')[col].shift(1)
    df[f'{col}_lag2'] = df.groupby('engine_no')[col].shift(2)

# Exponential moving average
for col in rolling_cols:
    df[f'{col}_ema'] = df.groupby('engine_no')[col].transform(lambda x: x.ewm(span=5, adjust=False).mean())

# Sensor trend (difference)
for col in rolling_cols:
    df[f'{col}_diff'] = df[col] - df[f'{col}_lag1']

# Binary label (classification: 1 if failure within 30 cycles)
df['label'] = df['time_to_failure'].apply(lambda x: 1 if x <= 30 else 0)

# Show sample rows
df[['engine_no', 'cycle', 'time_to_failure', 'label'] + [f'{col}_rolling_mean' for col in rolling_cols]].head()


   0   1       2       3      4       5       6        7        8      9   \
0   1   1 -0.0007 -0.0004  100.0  518.67  641.82  1589.70  1400.60  14.62   
1   1   2  0.0019 -0.0003  100.0  518.67  642.15  1591.82  1403.14  14.62   
2   1   3 -0.0043  0.0003  100.0  518.67  642.35  1587.99  1404.20  14.62   
3   1   4  0.0007  0.0000  100.0  518.67  642.35  1582.79  1401.87  14.62   
4   1   5 -0.0019 -0.0002  100.0  518.67  642.37  1582.85  1406.22  14.62   

   ...       18      19    20   21    22     23     24       25  26  27  
0  ...  8138.62  8.4195  0.03  392  2388  100.0  39.06  23.4190 NaN NaN  
1  ...  8131.49  8.4318  0.03  392  2388  100.0  39.00  23.4236 NaN NaN  
2  ...  8133.23  8.4178  0.03  390  2388  100.0  38.95  23.3442 NaN NaN  
3  ...  8133.83  8.3682  0.03  392  2388  100.0  38.88  23.3739 NaN NaN  
4  ...  8133.80  8.4294  0.03  393  2388  100.0  38.90  23.4044 NaN NaN  

[5 rows x 28 columns]


,engine_no,cycle,time_to_failure,label,sensor_2_rolling_mean,sensor_3_rolling_mean,sensor_5_rolling_mean
0,1,1,191,0,641.820000,1589.700000,14.62
1,1,2,190,0,641.985000,1590.760000,14.62
2,1,3,189,0,642.106667,1589.836667,14.62
3,1,4,188,0,642.167500,1588.075000,14.62
4,1,5,187,0,642.208000,1587.030000,14.62



### **Conclusion (Task 2 – Feature Engineering)**

In this task, we focused on **creating new features** from the existing dataset to improve our ability to **predict equipment failure**. These newly engineered features are not directly provided in the raw data but are derived to provide **more context and trends** over time.

---

### What we did:

1. **Calculated `time_to_failure`**:  
   - Shows how many cycles are left before each engine fails.
   - Helps in building a model that can predict *when* a failure might occur.

2. **Created a `label` column**:  
   - Used to classify whether an engine is close to failure (≤ 30 cycles left).
   - Useful for building classification models (0 = Safe, 1 = At risk of failure).

3. **Generated rolling mean features for key sensors**:  
   - Captures the **trend** of sensor behavior over the last few cycles.
   - Helps models understand patterns like slow degradation or sudden drops.

---

### How this helps answer the question:

- Raw sensor data only shows individual readings.
- With these **engineered features**, we give our model a better understanding of:
  - **How sensor values are changing over time**
  - **Whether the engine is approaching failure**
  - **What patterns lead to failure**

